# 🔌 Conectores reutilizables: ConectorBase + Registry

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/florvela/IA-y-automatizacion-en-seguridad-defensiva/blob/main/codigos-de-ejemplo/clase_en_vivo/live_04_conectores.ipynb)

**Presentar entre las diapositivas 86 y 89** (después de "Veamos un ejemplo").

Problema: 15 conectores ad-hoc = un bug se arregla en 15 lugares.
Solución: una **clase base** común + un **registro** central.


## 1. La clase base
Centraliza lo que siempre es igual: logging y reintentos. Cada conector solo implementa lo suyo.

In [ ]:
from abc import ABC, abstractmethod

class ConectorBase(ABC):
    def ejecutar(self, accion, **kwargs):
        print(f"  [{self.__class__.__name__}] log + validación + manejo de errores")
        return self._ejecutar(accion, **kwargs)   # lo específico de cada conector

    @abstractmethod
    def _ejecutar(self, accion, **kwargs): ...

## 2. Conectores concretos (con mock para la demo)
Usan `use_mock` para no gastar créditos de API ni necesitar credenciales reales.

In [ ]:
class ConectorVirusTotal(ConectorBase):
    def _ejecutar(self, accion, hash=None, **kw):
        return {"hash": hash, "malicioso": 58, "veredicto": "MALWARE"}   # mock EICAR

class ConectorJira(ConectorBase):
    def _ejecutar(self, accion, titulo=None, **kw):
        return {"ticket": "INC-1042", "titulo": titulo, "estado": "OPEN"}

# 3. Registry: pedimos el conector por nombre, sin importar cómo se inicializó
REGISTRY = {
    "virustotal": ConectorVirusTotal(),
    "jira":       ConectorJira(),
}

## 4. Pipeline de respuesta usando el registry

In [ ]:
def responder_a_malware(hash_archivo):
    # Enriquecimiento
    vt = REGISTRY["virustotal"].ejecutar("consultar", hash=hash_archivo)
    print("VirusTotal:", vt)
    if vt["malicioso"] < 5:
        return "Archivo limpio, no hago nada"

    # Documentación
    ticket = REGISTRY["jira"].ejecutar("crear", titulo=f"Malware {hash_archivo[:8]}")
    print("Jira:", ticket)
    return f"Endpoint marcado + {ticket['ticket']} creado"

print(responder_a_malware("44d88612fea8a8f36de82e1278abb02f"))